In [ ]:
# STRUCTURE:
# actual results, and initial stop (if never moved)
# no stop
# wide stop / max tolerable loss (20-40% range)
# 8, 10, 12.5, 15% stops
# 2, 2.5, 3 ATR stops
# maybe daily higher low and weekly higher low pivot stops, if I can figure that out, in a more advanced version

# ASSUMPTIONS:
# Entry is taken around the close of the day, so the first day is skipped when assessing R-multiples. Hence df.iloc[1:] syntax.
# On days where price gaps down below my stop, the stop is triggered at the open. Not always going to be the case
# No commissions or slippage on exit, so a round -1R loss if stop is hit intraday. Would like to fix in a future version.
# Intraday action is currently ignored. This may be an issue on days where my stop hits before a new high for the move. Will probably address with intraday price data and resampling. 

# FUTURE CONSIDERATIONS:
# Would I go about this process a different way with a larger dataset? Does pandas have a built in function, so I don't have to use the slower python loops?
# Instead of looping yfinance for every trade, should I store price data in a csv? Or would that be unnecessary? Depends on speed, data accuracy, etc. 
# Should I add some kind of drawdown calculation in a future version, so I can see what type of pullbacks I might expect and how viable wide stops actually are.
# While this isn't a consideration for v1, for future versions, I'd like to consider intraday stops, slippage, commissions,etc. I want to make this professional-grade.
# At some point I plan to test trailing stops on profitable trades to see what the most effective method there would be. E.g. pivot lows, moving average, atr, percentage trailing, etc.
# Worth adding MAE and MFE at some stage, to see how much a position goes for and against me before resolution. 

# EDGE CASES:
# Think about intraday entry and exit timing. E.g. what if my stop is below the low of the day, but I enter after the low of the day is set? Does it matter if entry is at the close?
# what if there's no data from df.iloc[1:]? 
# what should realised_r return if not stopped out? na value?
# what happens on a day if my stop hits before the high of the day? 

# TASKS:
# Consider changing max_r_baseline to a catastrophe stop (e.g. 30-50%) as this is a more realistic baseline. Could keep no_stop as max_r_no_stop for additional value. 
# Create loop for pct_stop variations. Use list as input...
# Create atr_stop functions.
# I think I should replace exit data and exit price with last_date and last_price (or final) and then add a boolean column for stopped_out (True/False), would help with available_r. 
# no_stop_max_r can be derived from this using the trade method's 1r value, if desired...
# Add realised_r (or available_r for open trades) and max_r back for each trade, and think about no_stop_max_r comparisons.
# I think max drawdown is going to be important when comparing stop loss types. E.g. A wider stop might look better on paper, but is the drop tolerable psychologically?

# ISSUES:
# Every time I update something like false_negative_test, I then have to go and update the other stop functions, which is a bit of an annoyance. May need to do something about that.
# Minor inconsistency in functions and order of ticker/date return.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

In [3]:
# pandas loop helper function

def trade_loop(data, entry_price, stop_price):

    max_price = entry_price
    exit_date = np.nan
    exit_price = np.nan

    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

        if stop_price >= row['Low']:
            if stop_price >= row['Open']:
                exit_price = row['Open']
                
            else:
                exit_price = stop_price
            
            exit_date = row_name.date()
            break

    if not pd.isna(exit_price):
        exit_price = round(exit_price, 2)

    max_pct = ((max_price - entry_price) / entry_price) * 100
    max_pct = round(max_pct, 2)

    return {'exit_date':exit_date, 
            'exit_price': exit_price, 
            'max_pct': max_pct}


In [4]:
# No stop helper function, returns max_price value in scenario with no stop loss

def no_stop(data, ticker, entry_date, entry_price, stop_price): 
    
    max_price = entry_price

    for row_name, row in data.iloc[1:].iterrows():

        if row['High'] > max_price:
            max_price = row['High']

    max_pct = ((max_price - entry_price) / entry_price) * 100

    return {'max_price':round(max_price, 2), 
            'max_pct':round(max_pct, 2)}

In [5]:
# False negative helper function, checks whether stop prevented trade from capturing sufficient portion of fat tail
def false_negative_test(max_pct, no_stop_max, pct_threshold, tail_threshold):

    if  (no_stop_max['max_pct']) < 0.01:
        tail_pct = np.nan
    else: 
        tail_pct = round((max_pct / no_stop_max['max_pct']) * 100, 2)

    if tail_pct > 100:
         tail_pct = 100
    
    if (no_stop_max['max_pct'] >= pct_threshold) and (tail_pct < tail_threshold):
            return {'bool':True, 
                    'tail_pct': tail_pct}

    return {'bool':False, 
            'tail_pct': tail_pct}

In [6]:
# Actual trade result function

def actual_outcome(data, trade_row, no_stop_max, pct_threshold, tail_threshold):

    max_price = trade_row['entry_price']
    
    date = pd.to_datetime(trade_row['entry_date']) + pd.DateOffset(days=1) # Possible calendar date issue to be aware of here. We want trading days...
    
    if not pd.isna(trade_row['exit_date']):
        for row_name, row in data.loc[date:trade_row['exit_date']].iterrows(): # Ideally would use iloc[1: ...] as elsewhere...

            if row['High'] > max_price:
                max_price = row['High']
    else:
        for row_name, row in data.iloc[1:].iterrows():

            if row['High'] > max_price:
                max_price = row['High']

    max_pct = ((max_price - trade_row['entry_price']) / trade_row['entry_price']) *100
    max_pct = round(max_pct, 2)

    false_negative = false_negative_test(max_pct, no_stop_max, pct_threshold, tail_threshold)

    return {'entry_date': trade_row['entry_date'], 
            'ticker': trade_row['ticker'], 
            'entry_price': trade_row['entry_price'],
            'stop_price': trade_row['stop_price'],
            'exit_date': trade_row['exit_date'],
            'exit_price': trade_row['exit_price'],
            'max_pct': max_pct,
            'no_stop_max_pct': no_stop_max['max_pct'],
            'tail_pct': false_negative['tail_pct'],
            'false_negative': false_negative['bool'],
            'stop_type': 'Actual Outcome'}

In [7]:
# Initial stop result function

def initial_stop(data, trade_row, no_stop_max, pct_threshold, tail_threshold):
    
    trade = trade_loop(data, trade_row['entry_price'], trade_row['stop_price'])

    false_negative = false_negative_test(trade['max_pct'], no_stop_max, pct_threshold, tail_threshold)

    return {'entry_date': trade_row['entry_date'], 
            'ticker': trade_row['ticker'], 
            'entry_price': trade_row['entry_price'],
            'stop_price': trade_row['stop_price'],
            'exit_date': trade['exit_date'],
            'exit_price': trade['exit_price'],
            'max_pct': trade['max_pct'],
            'no_stop_max_pct': no_stop_max['max_pct'],
            'tail_pct': false_negative['tail_pct'],
            'false_negative': false_negative['bool'],
            'stop_type': 'Initial Stop'}

In [8]:
# Percentage stop function

def pct_stop(data, trade_row, stop_pct, no_stop_max, pct_threshold, tail_threshold):

    stop_price = round(trade_row['entry_price'] * (1 - (stop_pct / 100)), 2)

    trade = trade_loop(data, trade_row['entry_price'], stop_price)

    false_negative = false_negative_test(trade['max_pct'], no_stop_max, pct_threshold, tail_threshold)

    return {'entry_date': trade_row['entry_date'], 
            'ticker': trade_row['ticker'], 
            'entry_price': trade_row['entry_price'],
            'stop_price': stop_price,
            'exit_date': trade['exit_date'],
            'exit_price': trade['exit_price'],
            'max_pct': trade['max_pct'],
            'no_stop_max_pct': no_stop_max['max_pct'],
            'tail_pct': false_negative['tail_pct'],
            'false_negative': false_negative['bool'],
            'stop_type': f'{stop_pct}_pct Stop'}

In [9]:
# Stop losses simulation function

def stop_sim(trade_row, stop_pct, pct_threshold, tail_threshold):
    
    data = yf.download(trade_row['ticker'], start=trade_row['entry_date'], multi_level_index=False, auto_adjust=True, progress=False)

    no_stop_max = no_stop(data, trade_row['ticker'], trade_row['entry_date'], trade_row['entry_price'], trade_row['stop_price'])

    sim_results = []
    sim_results.append(actual_outcome(data, trade_row, no_stop_max, pct_threshold, tail_threshold))
    sim_results.append(initial_stop(data, trade_row, no_stop_max, pct_threshold, tail_threshold))
    sim_results.append(pct_stop(data, trade_row, stop_pct, no_stop_max, pct_threshold, tail_threshold))

    return sim_results


In [36]:
# Reading trades CSV file, then using it as input for a list of dicts, which stores trade outputs.

trades = pd.read_csv('trades.csv')
stop_pct = 30
pct_threshold = 50
tail_threshold = 40

results = []

for row_name, row in trades.iterrows():
    results.extend(stop_sim(row, stop_pct, pct_threshold, tail_threshold))


In [13]:
# testing pct_stop loop 

list = [8, 10, 12.5, 15, 30]

results_test = []

trade_row = {

'entry_date': '2025-12-11',
'ticker': 'STX',
'entry_price': 308.34,
'stop_price': 277.5,
'exit_date': '2025-12-17',
'exit_price': 277.66
}

stop_pct = 30
pct_threshold = 50
tail_threshold = 40

data_test = yf.download(trade_row['ticker'], start=trade_row['entry_date'], multi_level_index=False, auto_adjust=True, progress=False)
no_stop_max = no_stop(data_test, trade_row['ticker'], trade_row['entry_date'], trade_row['entry_price'], trade_row['stop_price'])

for i in list:
        results_test.append(pct_stop(data_test, trade_row, i, no_stop_max, pct_threshold, tail_threshold))

results_test


[{'entry_date': '2025-12-11',
  'ticker': 'STX',
  'entry_price': 308.34,
  'stop_price': 283.67,
  'exit_date': datetime.date(2025, 12, 12),
  'exit_price': 283.67,
  'max_pct': 0.0,
  'no_stop_max_pct': np.float64(171.12),
  'tail_pct': np.float64(0.0),
  'false_negative': True,
  'stop_type': '8_pct Stop'},
 {'entry_date': '2025-12-11',
  'ticker': 'STX',
  'entry_price': 308.34,
  'stop_price': 277.51,
  'exit_date': datetime.date(2025, 12, 17),
  'exit_price': 277.51,
  'max_pct': 0.0,
  'no_stop_max_pct': np.float64(171.12),
  'tail_pct': np.float64(0.0),
  'false_negative': True,
  'stop_type': '10_pct Stop'},
 {'entry_date': '2025-12-11',
  'ticker': 'STX',
  'entry_price': 308.34,
  'stop_price': 269.8,
  'exit_date': nan,
  'exit_price': nan,
  'max_pct': np.float64(171.12),
  'no_stop_max_pct': np.float64(171.12),
  'tail_pct': np.float64(100.0),
  'false_negative': False,
  'stop_type': '12.5_pct Stop'},
 {'entry_date': '2025-12-11',
  'ticker': 'STX',
  'entry_price': 308.

In [37]:
# List of nested dicts converted into dataframe

results_df = pd.DataFrame(results)
results_df

,entry_date,ticker,entry_price,stop_price,exit_date,exit_price,max_pct,no_stop_max_pct,tail_pct,false_negative,stop_type
0,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.05,2.39,46.35,5.16,False,Actual Outcome
1,2025-08-12,TSLA,340.84,320.15,2025-08-20,320.15,2.39,46.35,5.16,False,Initial Stop
2,2025-08-12,TSLA,340.84,238.59,NaN,NaN,46.35,46.35,100.00,False,30_pct Stop
3,2025-08-26,STX,165.36,151.31,2025-11-21,229.72,79.36,378.96,20.94,True,Actual Outcome
4,2025-08-26,STX,165.36,151.31,NaN,NaN,378.96,378.96,100.00,False,Initial Stop
...,...,...,...,...,...,...,...,...,...,...,...
94,2026-04-08,LITE,896.23,761.51,NaN,NaN,13.92,13.92,100.00,False,Initial Stop
95,2026-04-08,LITE,896.23,627.36,NaN,NaN,13.92,13.92,100.00,False,30_pct Stop
96,2026-04-08,STX,495.76,436.84,NaN,NaN,59.76,59.76,100.00,False,Actual Outcome
97,2026-04-08,STX,495.76,436.84,NaN,NaN,59.76,59.76,100.00,False,Initial Stop


In [ ]:
# Using pandas functionality to output dataframe to CSV

results_df.to_csv('results.csv', index=False)